<a href="https://colab.research.google.com/github/Mmbsaksd/Full_Stack_GenerativeAI_And_Agentic_AI/blob/main/Complete_Fine_Tuning_Non_Instruction%2C_Instruction%2C_and_Preference_Data_with_Huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Stage 1:** Non-instruction Causal LLM Fine-tuning or Domain-Adaptive Continued Pretraining

In [1]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.0 MB/s eta 0:00:00


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge (1).pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/content/pharma_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/content/pharma_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1


In [4]:
config = Config()

In [5]:
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge (1).pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [6]:
import json
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge (1).pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [7]:
config.output_dir

'/content/pharma_tinyllama_lora_output'

In [8]:
config.processed_data_dir

'/content/pharma_processed_data'

In [9]:
import os
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [10]:
# ============================================================
# Hugging Face repo names
# ============================================================

HF_USERNAME = "mmbsaksd"

BASE_MODEL_NAME = config.model_name

# Stage 1: Non-instruction LoRA adapter
HF_REPO_NON_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-lora-adapter"

# Stage 1 merged model
HF_REPO_NON_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-non-instruction-merged"

# Stage 2: Instruction LoRA adapter
HF_REPO_INSTRUCTION_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-instruction-lora-adapter"

# Stage 2 merged model
HF_REPO_INSTRUCTION_MERGED = f"{HF_USERNAME}/pharma-tinyllama-instruction-merged"

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"{HF_USERNAME}/pharma-tinyllama-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"{HF_USERNAME}/pharma-tinyllama-dpo-merged"

print(HF_REPO_NON_INSTRUCTION_ADAPTER)
print(HF_REPO_INSTRUCTION_ADAPTER)
print(HF_REPO_DPO_ADAPTER)

mmbsaksd/pharma-tinyllama-non-instruction-lora-adapter
mmbsaksd/pharma-tinyllama-instruction-lora-adapter
mmbsaksd/pharma-tinyllama-dpo-lora-adapter


In [11]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge (1).pdf


In [12]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [13]:
config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge (1).pdf'

In [14]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [15]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [16]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

In [17]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [18]:
cleaned_pages = []

In [19]:
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [20]:
print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 6


In [21]:
print(cleaned_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenous cholesterol synthesis

In [22]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [23]:
paragraph_records = split_into_paragraph_records(cleaned_pages)

In [24]:
print("Total paragraph records:", len(paragraph_records))

Total paragraph records: 9


In [25]:
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Page: 1 | Paragraph: 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Page: 1 | Paragraph: 2 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin

In [26]:
# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_process.jsonl


In [27]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================
from datasets import Dataset
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )
text_dataset = Dataset.from_list(paragraph_records)

In [28]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [29]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


In [30]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)


In [31]:
split_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    test: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})

In [32]:
from datasets import DatasetDict
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


In [33]:
# ============================================================
# 11. Load tokenizer
# ============================================================

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [34]:
tokenizer.eos_token

'</s>'

In [35]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


In [36]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [37]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

In [38]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

In [39]:
#tokenized_datasets['train']['input_ids'][0]

In [40]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)


    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }


    input_id_blocks = []
    attention_mask_blocks = []

    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }


In [41]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

In [42]:
sample = final_dataset["train"][0]

In [43]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

In [44]:
# ============================================================
# 13. Load base model
# ============================================================
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [45]:
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()


In [46]:
from transformers import AutoModelForCausalLM

if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


In [47]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [48]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [49]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [50]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [51]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

from transformers import TrainingArguments
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)
from transformers import TrainingArguments
training_args = TrainingArguments(**training_kwargs)

In [52]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


In [53]:
import warnings
warnings.filterwarnings("ignore")

In [54]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

Step,Training Loss
1,2.148971
2,2.148971
3,2.124020


Training completed.


In [55]:
for log in trainer.state.log_history:
    print(log)

{'loss': 2.14897084236145, 'grad_norm': 0.6881399154663086, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 2.1489710807800293, 'grad_norm': 0.6994675993919373, 'learning_rate': 4e-05, 'epoch': 2.0, 'step': 2}
{'loss': 2.1240196228027344, 'grad_norm': 0.673311173915863, 'learning_rate': 8e-05, 'epoch': 3.0, 'step': 3}
{'train_runtime': 13.3002, 'train_samples_per_second': 1.353, 'train_steps_per_second': 0.226, 'total_flos': 57901993426944.0, 'train_loss': 2.1406538486480713, 'epoch': 3.0, 'step': 3}


In [56]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

('/content/pharma_tinyllama_lora_adapter/tokenizer_config.json',
 '/content/pharma_tinyllama_lora_adapter/tokenizer.json')

In [57]:
print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'README.md']


In [58]:
HF_REPO_NON_INSTRUCTION_ADAPTER

'mmbsaksd/pharma-tinyllama-non-instruction-lora-adapter'

In [59]:
# ============================================================
# Push Stage 1 non-instruction LoRA adapter to Hugging Face
# ============================================================

# trainer.model.push_to_hub(
#     HF_REPO_NON_INSTRUCTION_ADAPTER,
#     private=True
# )


# tokenizer.push_to_hub(
#     HF_REPO_NON_INSTRUCTION_ADAPTER,
#     private=True
# )

# print("Stage 1 non-instruction LoRA adapter pushed to:")
# print(HF_REPO_NON_INSTRUCTION_ADAPTER)

In [60]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [61]:
from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [62]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [63]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

In [64]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [65]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


In [66]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [67]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Notebook 2, we will create instruction prompts for Q&A.

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]


In [68]:
# ============================================================
# 23. Test text continuation
# ============================================================

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is a sulfonylurea class drug, which has been used to treat type 2 diabetes for over 30 years. However, despite its proven efficacy and safety, the mechanism by which it works remains unknown. A recent study from the National Institutes of Health (NIH) reveals that Metformin can directly affect the cellular processes that regulate the production of insulin. This finding could lead to the development of new drugs targeting these processes.
"These findings are very important because they suggest that Met

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a statin drug that reduces cholesterol and LDL cholesterol, lowers LDL-C by 39% compared to statins alone.
The study was published online in the New England Journal of Medicine.
Atorvastatin is a lipid (fat) lowering drug used for primary prevention of cardiovascular disease, as well as secondary prevention of coronary artery disease, strokes and other types of heart attacks.
Atrial fibrillation is an irregular heartbeat that causes unpredict

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery, reducing time-to-market and increasing the efficiency of R&D operations.
Industrial IoT is driving the next wave of innovation in healthcare as a growing number of devices and apps are connected to create smart environments for better patient care.
MedTech IoT is revol

In [69]:
# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

In [70]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [71]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)

In [72]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [73]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model


## **Stage 2:** Continue with Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model

In [74]:
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"

In [75]:
from datasets import load_dataset

In [76]:
instruction_dataset = load_dataset(
    "json",
    data_files=instruction_data_path,
    split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [77]:
print(instruction_dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic'],
    num_rows: 48
})


In [78]:
print(instruction_dataset[0])

{'instruction': 'Explain the primary mechanism of action of metformin.', 'input': '', 'output': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [79]:
# ============================================================
# Format instruction records
# ============================================================
# We convert every record into Alpaca-style training text.

def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

In [80]:
instruction_dataset = instruction_dataset.map(format_instruction_record)

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

In [81]:
instruction_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
    num_rows: 48
})

In [82]:
print(instruction_dataset[0]["text"])

### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


In [83]:
# ============================================================
# Create train-validation split
# ============================================================

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 8
    })
})
Train examples: 40
Validation examples: 8


In [84]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
# The tokenizer converts text into token IDs for model training.

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [85]:
instruction_max_length = 512

In [86]:
def tokenize_instruction_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Ignore padding tokens in the loss calculation.
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

In [87]:
instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
})


In [88]:
# ============================================================
# Load merged Stage 1 model and add new LoRA adapter for instruction tuning
# ============================================================

# Merged Stage 1 Model
#    +
# New LoRA adapter for instruction tuning

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "/content/pharma_tinyllama_merged_model"


In [89]:
if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [90]:
instruction_base_model.config.use_cache = False

# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [91]:
# ============================================================
# Instruction fine-tuning data collator
# ============================================================
# This prepares mini-batches for causal language model training.

from transformers import DataCollatorForLanguageModeling
instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [92]:
# ============================================================
# Instruction fine-tuning arguments
# ============================================================

instruction_output_dir = "/content/pharma_tinyllama_instruction_lora_output"
instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

In [93]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,

    # Training will stop after 5 optimizer steps.
    # max_steps overrides num_train_epochs.
    num_train_epochs=5,
    max_steps=5,

    # Batch settings.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings.
    learning_rate=1e-4,
    warmup_steps=2,
    weight_decay=0.01,

    # Show training loss at every step.
    logging_steps=1,
    logging_first_step=True,

    # Run validation at every step.
    eval_strategy="steps",
    eval_steps=1,

    # Save checkpoint at final step.
    save_steps=5,
    save_total_limit=2,

    # Precision settings.
    fp16=use_cuda,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,
)

print(instruction_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1,
eval_strategy=IntervalStrategy.STEPS,
eval_us

In [94]:
# ============================================================
# Build instruction Trainer
# ============================================================

from transformers import Trainer
instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_training_args,
    train_dataset=instruction_tokenized_datasets["train"],
    eval_dataset=instruction_tokenized_datasets["validation"],
    data_collator=instruction_data_collator,
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


In [95]:
# ============================================================
# Start instruction fine-tuning
# ============================================================

instruction_train_result = instruction_trainer.train()


Step,Training Loss,Validation Loss
1,2.055867,2.298050
2,2.167667,2.253776
3,2.477807,2.167026
4,2.052778,2.111385
5,2.101425,2.084243


In [96]:
print("Instruction fine-tuning completed.")
print(instruction_train_result)

Instruction fine-tuning completed.
TrainOutput(global_step=5, training_loss=2.171108675003052, metrics={'train_runtime': 23.9946, 'train_samples_per_second': 1.667, 'train_steps_per_second': 0.208, 'total_flos': 128671096504320.0, 'train_loss': 2.171108675003052, 'epoch': 1.0})


In [97]:
# ============================================================
# Save final instruction-tuned LoRA adapter
# ============================================================
# This adapter now contains Stage 1 domain adaptation + Stage 2 instruction tuning.

import os

instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"
os.makedirs(instruction_adapter_dir, exist_ok=True)

instruction_trainer.model.save_pretrained(instruction_adapter_dir)
tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(os.listdir(instruction_adapter_dir))

Final instruction-tuned LoRA adapter saved to: /content/pharma_tinyllama_instruction_lora_adapter
['tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer.json', 'README.md']


In [98]:
# ============================================================
# Push Stage 2 instruction LoRA adapter to Hugging Face
# ============================================================

# instruction_trainer.model.push_to_hub(
#     HF_REPO_INSTRUCTION_ADAPTER,
#     private=True
# )

# tokenizer.push_to_hub(
#     HF_REPO_INSTRUCTION_ADAPTER,
#     private=True
# )

# print("Stage 2 instruction LoRA adapter pushed to:")
# print(HF_REPO_INSTRUCTION_ADAPTER)

In [99]:
# ============================================================
# Reload final instruction-tuned adapter for inference
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
         merged_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final instruction-tuned model loaded successfully.


In [100]:
# ============================================================
# Instruction-style inference helper
# ============================================================

def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )


In [101]:
def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_instruction_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(final_instruction_model.device)

    with torch.no_grad():
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [102]:
# ============================================================
# Test instruction-tuned pharma model
# ============================================================

test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin acts by blocking the conversion of glucose to glucolipotion in the liver. This results in a decrease in the blood glucose level and the formation of ketone bodies, which is what occurs in diabetes mellitus.

QUESTION:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

### Response:
The combination of atorvastatin and ezetimibe has been shown to be superior in the treatment of patients with hypercholesterolemia than either drug alone. Atorvastatin, a HMG-CoA reductase inhibitor, lowers triglycerides and high-density lipoprotein cholesterol (HDL-C) and increases low-density lipoprotein cholesterol (LDL-C). Ezetimibe is an inhibitor of the JAK2 pathway that reduces intravascular lipids by increasing absorption of dietary fat and decreasing the synthesis of chylomicrons. The JAK
QUESTION:
Summarize the role of lipid nanoparticles in mRNA vaccines.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Summarize the role of lipid nanoparticles in mRNA vaccines.

### Response:
- Lipid Nanoparticles are very small particles that can be used to deliver a gene into cells, this process is called gene therapy. The particles are able to move through blood and transport their cargo through the body. 
- Lipid Nanoparticles are able to carry genes to different tissues which will allow them to be incorporated into the cellular structure. This process is called gene transfer.

### Instruction:
Describe the process of transcription.

### Response:
- Transcription is the process of converting RNA to DNA. The first step is the binding of DNA to RNA (Gene to DNA) then the DNA is copied creating a double stranded DNA mole
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:
### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:

There are many reasons for why the drug discovery process ne

In [103]:
# ============================================================
# Merge instruction-tuned LoRA adapter into base model
# ============================================================
# This creates a standalone instruction-tuned model.
# Later, we can use this merged model as the base model for preference tuning.

import os
import gc
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Path where the final merged instruction-tuned model will be saved.
merged_instruction_model_dir = "/content/pharma_tinyllama_instruction_merged_model"

os.makedirs(merged_instruction_model_dir, exist_ok=True)

In [104]:
# Load the original base model in normal precision for safe merging.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    merged_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Load the tokenizer.
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
)

if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

# Attach the final instruction-tuned LoRA adapter.
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_adapter_dir,
)

# Merge LoRA adapter weights into the base model weights.
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

# Save the standalone merged model and tokenizer.
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
tokenizer_for_merge.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model saved to: /content/pharma_tinyllama_instruction_merged_model


## **Stage 3:** Preference Tuning with DPO

In [105]:
# ============================================================
# 25. Install TRL for DPO training
# ============================================================
# TRL provides DPOTrainer and DPOConfig for preference tuning.

!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 14.4 MB/s eta 0:00:00


In [106]:
# ============================================================
# 26. Load DPO preference dataset
# ============================================================
# Expected columns: prompt, chosen, rejected

from datasets import load_dataset

preference_data_path = "/content/pharma_preference_dataset.jsonl"

preference_dataset = load_dataset(
    "json",
    data_files=preference_data_path,
    split="train"
)

print(preference_dataset)
print(preference_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})
{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [107]:
preference_dataset = preference_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

preference_dataset["validation"] = preference_dataset.pop("test")

print("After train-validation split:")
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

After train-validation split:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 8
    })
})
Train rows: 40
Validation rows: 8


In [108]:
merged_instruction_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [109]:
# ============================================================
# Load merged instruction model as base for preference tuning
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

use_cuda = torch.cuda.is_available()

if use_cuda:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )

    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_base_model.config.use_cache = False

# Create a new LoRA adapter for preference tuning.
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [110]:
# ============================================================
# 30. Configure DPO training
# ============================================================

import os
#import inspect
#from transformers import TrainingArguments
from trl import DPOTrainer
from trl import DPOConfig

try:
    from trl import DPOConfig
    has_dpo_config = True
except ImportError:
    DPOConfig = None
    has_dpo_config = False

preference_output_dir = "/content/pharma_tinyllama_preference_dpo_output"
preference_adapter_dir = "/content/pharma_tinyllama_preference_dpo_lora_adapter"

os.makedirs(preference_output_dir, exist_ok=True)
os.makedirs(preference_adapter_dir, exist_ok=True)

In [111]:
# ============================================================
# 30. Create DPO training arguments - Simple Version
# ============================================================

from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir=preference_output_dir,

    # Training duration
    num_train_epochs=3,
    max_steps=5,

    # Batch settings
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizer settings
    learning_rate=5e-5,
    warmup_steps=2,
    weight_decay=0.01,

    # Logging and evaluation
    logging_steps=1,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=1,

    # Checkpoint saving
    save_steps=5,
    save_total_limit=2,

    # Precision settings
    fp16=False,
    bf16=False,

    # Disable external logging tools
    report_to="none",

    # Keep required columns
    remove_unused_columns=False,

    # DPO hyperparameter
    beta=0.1,
)

print(dpo_training_args)

DPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.1,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat

In [112]:
# ============================================================
# 31. Build DPOTrainer - Simple Student Version
# ============================================================

from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,

    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],

    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DPOTrainer is ready.


In [113]:
# ============================================================
# 32. Start DPO preference tuning
# ============================================================

dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693147,0.693147,2.127015,1413.000000,-3.540191,-3.540112,0.556302,0.000000,0.000000,0.000000,0.000000,-127.232809,-120.171694
2,0.693147,0.651014,2.124515,2789.000000,-3.541246,-3.542329,0.556880,0.023871,-0.063038,1.000000,0.086909,-126.994102,-120.802072
3,0.659902,0.534184,2.119263,4051.000000,-3.542691,-3.545710,0.563244,0.070084,-0.286468,1.000000,0.356552,-126.531971,-123.036372
4,0.545496,0.466362,2.114380,5380.000000,-3.543301,-3.547385,0.564715,0.101267,-0.437332,1.000000,0.538599,-126.220140,-124.545017
5,0.471567,0.435623,2.111933,6698.000000,-3.543747,-3.548643,0.564715,0.115987,-0.511540,1.000000,0.627527,-126.072939,-125.287095


DPO preference tuning completed.
TrainOutput(global_step=5, training_loss=0.6126518428325654, metrics={'train_runtime': 36.9081, 'train_samples_per_second': 1.084, 'train_steps_per_second': 0.135, 'total_flos': 49278084096000.0, 'train_loss': 0.6126518428325654, 'epoch': 1.0})


In [114]:
# ============================================================
# 33. Save DPO preference-tuned LoRA adapter
# ============================================================

dpo_trainer.model.save_pretrained(preference_adapter_dir)
tokenizer.save_pretrained(preference_adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_adapter_dir}")
print(os.listdir(preference_adapter_dir))


Preference-tuned LoRA adapter saved to: /content/pharma_tinyllama_preference_dpo_lora_adapter
['tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'ref', 'tokenizer.json', 'README.md']


In [115]:
# ============================================================
# Push Stage 3 DPO LoRA adapter to Hugging Face
# ============================================================

# dpo_trainer.model.push_to_hub(
#     HF_REPO_DPO_ADAPTER,
#     private=True
# )

# tokenizer.push_to_hub(
#     HF_REPO_DPO_ADAPTER,
#     private=True
# )

# print("Stage 3 DPO LoRA adapter pushed to:")
# print(HF_REPO_DPO_ADAPTER)

In [116]:
# ============================================================
# 34. Reload preference-tuned model for inference
# ============================================================

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

preference_inference_model = PeftModel.from_pretrained(
    preference_inference_base_model,
    preference_adapter_dir,
)

preference_inference_model.eval()

print("Preference-tuned model loaded successfully for inference.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preference-tuned model loaded successfully for inference.


In [117]:
# ============================================================
# 35. Preference-tuned inference helper
# ============================================================

def build_preference_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [118]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_preference_prompt(instruction, input_text)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [119]:
# ============================================================
# 36. Test preference-tuned pharma model
# ============================================================

preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

for question in preference_test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_preference_response(question, max_new_tokens=150))


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin has been shown to inhibit glucose uptake into cells and to reduce hepatic gluconeogenesis by blocking the activity of the enzymes gluconeogenic flux and gluconeogenic protein synthesis. It also reduces fatty acid production in adipocytes, thereby increasing energy expenditure. Metformin is a first-generation orexin receptor antagonist, which leads to reduced food intake and weight loss, with no significant side effects. Metformin is the only drug approved for type 2 diabetes mellitus by the FDA and by the European Medicines Agency.

### Instruction:
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
The purpose of the study is to examine whether there are differences between the predicted AI and experimental data. In order to do this, a new dataset needs to be created from the drug discovery literature. 

The drugs will be categorized into four groups based on their mechanism of action (MOA). The four groups will be:
- Compounds with unknown MOA (Group 1)
- Moderately known MOAs (Group 2)
- Known MOAs (Group 3)
- Fully known MOAs (Group 4)

We have developed a new methodology for classification called 3-class classification where we predict if the compound has been classified as Group 1
QUESTION:
Define pharmacovigilance.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Define pharmacovigilance.

### Response:
The pharmacovigilance is the systematic study of safety of drugs, devices and other health-related products in order to identify, monitor and prevent adverse events and incidents (safety issues) that may occur during their development, testing, marketing and use. Pharmacovigilance studies are not limited only to drugs but also includes medical devices and other health-related products, including biological products, diagnostics, cosmetics and veterinary medicines. The term "vigilance" was introduced by the European Commission in 2013 as a way of distinguishing between the term "drug safety" and "pharmacovigilance". Pharm
QUESTION:
Explain why pharmacovigilance continues after drug approval.

MODEL RESPONSE:
### Instruction:
Explain why pharmacovigilance continues after drug approval.

### Response:
Pharmacovigilance is an important aspect of the entire drug development process, and its goal is to ensure that a drug does not caus

In [120]:
# ============================================================
# 37. Optional: Merge DPO preference adapter into the instruction-tuned base model
# ============================================================
# Use this only after preference tuning is complete and you want a standalone final model.

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_merged_preference_model_dir = "/content/pharma_tinyllama_final_preference_merged_model"
os.makedirs(final_merged_preference_model_dir, exist_ok=True)

In [121]:
# Load the merged instruction model in normal precision for safe merging.
base_model_for_preference_merge = AutoModelForCausalLM.from_pretrained(
    merged_instruction_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Attach the DPO preference LoRA adapter.
model_with_preference_adapter = PeftModel.from_pretrained(
    base_model_for_preference_merge,
    preference_adapter_dir,
)

# Merge the preference adapter into the instruction-tuned base model.
final_merged_preference_model = model_with_preference_adapter.merge_and_unload()

# Save final standalone model and tokenizer.
final_merged_preference_model.save_pretrained(final_merged_preference_model_dir)
tokenizer.save_pretrained(final_merged_preference_model_dir)

print(f"Final merged preference-tuned model saved to: {final_merged_preference_model_dir}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final merged preference-tuned model saved to: /content/pharma_tinyllama_final_preference_merged_model
